# ETHUSDT — Progressive Reinvestment Parameter Optimisation

Grid search over start_pct, increment, and cap to find the best progressive reinvestment strategy for ETHUSDT with 70% ATH sells.

In [ ]:
import pandas as pd
import requests
import time
import json
import numpy as np
from datetime import datetime, timezone
import itertools

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [ ]:
def fetch_monthly_klines(symbol='ETHUSDT', limit=500):
    url = 'https://api.binance.com/api/v3/klines'
    all_klines = []
    start_time = 0
    while True:
        params = {'symbol': symbol, 'interval': '1M', 'startTime': start_time, 'limit': limit}
        resp = requests.get(url, params=params)
        data = resp.json()
        if not data or len(data) == 0:
            break
        all_klines.extend(data)
        print(f'Fetched {len(data)} months, up to {datetime.fromtimestamp(data[-1][0]/1000, tz=timezone.utc).strftime("%Y-%m")}')
        if len(data) < limit:
            break
        start_time = data[-1][0] + 1
        time.sleep(0.1)
    return all_klines

klines = fetch_monthly_klines()

columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume',
           'close_time', 'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore']
df = pd.DataFrame(klines, columns=columns)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
for c in ['open', 'high', 'low', 'close', 'volume']:
    df[c] = df[c].astype(float)
df = df[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
df = df.sort_values('date').reset_index(drop=True)
print(f'{df["date"].min().strftime("%Y-%m")} → {df["date"].max().strftime("%Y-%m")} ({len(df)} months)')

In [ ]:
def run_backtest(start_pct, increment, cap):
    btc_held = 0.0; usdt_reserve = 0.0; total_invested = 0.0; highest_close = 0.0
    reinvest_pct = start_pct
    records = []

    for i, row in df.iterrows():
        close = row['close']
        if close < row['open']:
            extra = usdt_reserve * (reinvest_pct / 100.0) if usdt_reserve > 0 else 0.0
            total_usdt = 10.0 + extra
            btc_held += total_usdt / close
            total_invested += 10.0
            usdt_reserve -= extra

        prev_highest = highest_close
        if close > highest_close:
            highest_close = close

        if btc_held > 0.000001 and close > prev_highest:
            sell_btc = btc_held * 0.7
            usdt_reserve += sell_btc * close
            btc_held -= sell_btc
            reinvest_pct = start_pct
        else:
            if reinvest_pct < cap:
                reinvest_pct = min(cap, reinvest_pct + increment)

        records.append(btc_held * close + usdt_reserve)

    final_val = records[-1]
    ret_pct = (final_val / total_invested - 1) * 100

    eq = pd.Series(records)
    running_max = eq.cummax()
    dd = ((running_max - eq) / running_max).max() * 100

    monthly_returns = eq.pct_change().dropna()
    monthly_returns = monthly_returns[monthly_returns.index >= 12]
    sharpe = (monthly_returns.mean() / monthly_returns.std()) * np.sqrt(12) if monthly_returns.std() > 0 else 0.0

    pos_sum = monthly_returns[monthly_returns > 0].sum()
    neg_sum = monthly_returns[monthly_returns < 0].abs().sum()
    pf = pos_sum / neg_sum if neg_sum > 0 else float('inf')

    annualized_ret = (1 + ret_pct / 100) ** (12 / len(df)) - 1
    calmar = annualized_ret / (dd / 100) if dd > 0 else 0

    return {
        'start': start_pct, 'inc': increment, 'cap': cap,
        'label': f'{start_pct}%+{increment}%->{cap}%',
        'return_pct': ret_pct, 'max_dd': dd, 'sharpe': sharpe,
        'calmar': calmar, 'pf': pf, 'final_val': final_val, 'btc': btc_held,
    }

starts = [1, 3, 5, 10, 15]
increments = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
caps = [30, 40, 50, 60, 70, 80, 90, 100]

print(f'Testing {len(starts) * len(increments) * len(caps)} combinations...')
results_list = []
for start, inc, cap in itertools.product(starts, increments, caps):
    r = run_backtest(start, inc, cap)
    results_list.append(r)

opt = pd.DataFrame(results_list)
print(f'Done. {len(opt)} results.')

In [ ]:
# Top 20 by Calmar

opt.nlargest(20, 'calmar')[['label', 'return_pct', 'max_dd', 'sharpe', 'calmar', 'pf']]

In [ ]:
# Top 20 by Sharpe

opt.nlargest(20, 'sharpe')[['label', 'return_pct', 'max_dd', 'sharpe', 'calmar', 'pf']]

In [ ]:
# Top 20 by return (DD < 50%)

opt[opt['max_dd'] < 50].nlargest(20, 'return_pct')[['label', 'return_pct', 'max_dd', 'sharpe', 'calmar', 'pf']]

In [ ]:
# Overall best by each metric

best_calmar = opt.loc[opt['calmar'].idxmax()]
best_sharpe = opt.loc[opt['sharpe'].idxmax()]
best_return = opt.loc[opt['return_pct'].idxmax()]

print("="*65)
print("ETH OPTIMUM BY EACH METRIC")
print("="*65)
print(f"By Calmar:  {best_calmar['label']:>15s}  Return={best_calmar['return_pct']:6.1f}%  DD={best_calmar['max_dd']:5.1f}%  Sharpe={best_calmar['sharpe']:.2f}  Calmar={best_calmar['calmar']:.2f}")
print(f"By Sharpe:  {best_sharpe['label']:>15s}  Return={best_sharpe['return_pct']:6.1f}%  DD={best_sharpe['max_dd']:5.1f}%  Sharpe={best_sharpe['sharpe']:.2f}  Calmar={best_sharpe['calmar']:.2f}")
print(f"By Return:  {best_return['label']:>15s}  Return={best_return['return_pct']:6.1f}%  DD={best_return['max_dd']:5.1f}%  Sharpe={best_return['sharpe']:.2f}  Calmar={best_return['calmar']:.2f}")

print()

# Best with DD < 40%
print("Best with DD < 40%:")
good = opt[opt['max_dd'] < 40].nlargest(10, 'calmar')
for _, r in good.iterrows():
    print(f"  {r['label']:>15s}  Return={r['return_pct']:6.1f}%  DD={r['max_dd']:5.1f}%  Sharpe={r['sharpe']:.2f}  Calmar={r['calmar']:.2f}")

In [ ]:
# Compare with BTC optimum and no-reinvest baselines

print("="*65)
print("BEST ETH PARAMS vs BTC OPTIMUM on ETH")
print("="*65)

# Run BTC optimum (1%+2%->50) on ETH
btc_opt = run_backtest(1, 2, 50)
print(f"BTC optimum on ETH:   Return={btc_opt['return_pct']:.1f}%  DD={btc_opt['max_dd']:.1f}%  Sharpe={btc_opt['sharpe']:.2f}  Calmar={btc_opt['calmar']:.2f}")

# Run ETH best by Calmar
ec = best_calmar
print(f"ETH best by Calmar:   Return={ec['return_pct']:.1f}%  DD={ec['max_dd']:.1f}%  Sharpe={ec['sharpe']:.2f}  Calmar={ec['calmar']:.2f}")

# Run ETH best by Sharpe
es = best_sharpe
print(f"ETH best by Sharpe:   Return={es['return_pct']:.1f}%  DD={es['max_dd']:.1f}%  Sharpe={es['sharpe']:.2f}  Calmar={es['calmar']:.2f}")

# Run no reinvest (just $10)
no_reinv = run_backtest(0, 0, 0)
no_reinv2 = {'return_pct': 0, 'max_dd': 0}  # placeholder, recompute
btc = 0; usdt = 0; inv = 0; hc = 0
for i, row in df.iterrows():
    c = row['close']
    if c < row['open']:
        btc += 10 / c
        inv += 10
    ph = hc
    if c > hc: hc = c
    if btc > 0.000001 and c > ph:
        usdt += btc * 0.7 * c
        btc -= btc * 0.7
fv = btc * df.iloc[-1]['close'] + usdt
ret = (fv / inv - 1) * 100
print(f"No reinvest ($10 only): Return={ret:.1f}%  Final=${fv:.0f}")

In [ ]:
# Heatmaps

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, (metric, title, cmap) in enumerate([
    ('return_pct', 'Return (%)', 'RdYlGn'),
    ('max_dd', 'Max Drawdown (%)', 'RdYlGn_r'),
    ('sharpe', 'Sharpe Ratio', 'RdYlGn'),
    ('calmar', 'Calmar Ratio', 'RdYlGn')
]):
    ax = axes[idx // 2][idx % 2]
    pivot = opt[opt['cap'] == 50].pivot_table(
        index='start', columns='inc', values=metric, aggfunc='mean'
    )
    im = ax.imshow(pivot.values, cmap=cmap, aspect='auto', interpolation='nearest')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{int(c)}%' for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{int(r)}%' for r in pivot.index])
    ax.set_xlabel('Increment per buy')
    ax.set_ylabel('Starting %')
    ax.set_title(f'{title} (cap=50%)')
    plt.colorbar(im, ax=ax)

plt.suptitle('ETHUSDT — Progressive Reinvestment Heatmaps (cap=50%)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dca-bt-buy-bot-eth-optimisation.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-eth-optimisation.png')
plt.show()